# Check data quality

Quality checks identify missing reference dates, privacy review needs, and split leakage before annotation or training. The data contract in `docs/interface.md` defines the fields and grouping rules.

The default dataset is synthetic. Its plots check the workflow and do not support research conclusions. Change `DATASET` in the setup cell after collecting and preparing a reviewed corpus.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from smartnotice.demo import run_demo
from smartnotice.io import read_jsonl
from smartnotice.schemas import Notice

%matplotlib inline

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists(), (
    "Open this notebook from the repository root or notebooks/"
)

DATASET = "demo"  # Set to "research" to read the prepared local corpus.
if DATASET == "demo":
    run_demo(ROOT)
    NOTICE_PATH = ROOT / "data/interim/demo/notices.jsonl"
    SPLIT_PATH = ROOT / "data/processed/demo/splits.jsonl"
    ANNOTATION_PATH = ROOT / "data/annotations/examples/events.jsonl"
    PAIR_PATH = ROOT / "data/processed/demo/applicability.jsonl"
else:
    NOTICE_PATH = ROOT / "data/interim/notices.jsonl"
    SPLIT_PATH = ROOT / "data/processed/splits.jsonl"
    ANNOTATION_PATH = ROOT / "data/annotations/adjudicated.jsonl"
    PAIR_PATH = ROOT / "data/processed/applicability.jsonl"

FIGURES = ROOT / "reports/generated" / DATASET / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
notices = [Notice.model_validate(row) for row in read_jsonl(NOTICE_PATH)]
assert notices, "Prepare at least one notice before analysis"
frame = pd.DataFrame([n.model_dump(mode="json") for n in notices])
print(f"Dataset: {DATASET}; documents: {len(notices)}; synthetic: {int(frame.synthetic.sum())}")
if DATASET == "research":
    assert not frame.synthetic.any(), "Exclude synthetic fixtures from research measurements"

In [ ]:
from smartnotice.quality import quality_summary

splits = read_jsonl(SPLIT_PATH)
summary = quality_summary(notices, splits)
display(
    pd.Series(
        {k: v for k, v in summary.items() if not isinstance(v, (dict, list))}, name="value"
    ).to_frame()
)
assert not summary["split_group_leaks"], "Related groups cross evaluation boundaries"
assert summary["empty_documents"] == 0

## Inspect privacy and dates

Automatic masking does not establish anonymity. Review documents flagged for privacy before sharing text or annotation tasks. Missing publication dates must not be replaced by collection dates.

In [ ]:
display(
    frame[
        [
            "doc_id",
            "source_type",
            "reference_date",
            "reference_date_origin",
            "redaction_count",
            "privacy_review_required",
        ]
    ]
)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
frame.groupby("source_type").reference_date.apply(lambda values: values.isna().sum()).plot.bar(
    ax=axes[0], title="Missing reference dates"
)
frame.groupby("source_type").redaction_count.sum().plot.bar(
    ax=axes[1], title="Automatic redactions"
)
for axis in axes:
    axis.set_xlabel("Source")
    axis.set_ylabel("Count")
    axis.tick_params(axis="x", rotation=0)
fig.tight_layout()
fig.savefig(FIGURES / "quality_dates_privacy.png", dpi=160)
plt.show()
plt.close("all")

## Inspect duplicate groups and split balance

The allocator targets proportions by document count. Review large groups and empty splits instead of moving related documents across boundaries to fill quotas.

In [ ]:
assigned = pd.DataFrame(splits)
joined = frame.merge(assigned, on="doc_id", suffixes=("_source", "_split"), validate="one_to_one")
display(pd.crosstab(joined["split"], joined["source_type"]))
display(
    joined.groupby("group_id_split")
    .agg(documents=("doc_id", "size"), splits=("split", lambda s: ", ".join(sorted(set(s)))))
    .sort_values("documents", ascending=False)
)
assert not joined.loc[joined["split"] == "domain_test", "source_type"].ne("web").any()
print("Exact duplicate prepared texts:", int(frame.text.duplicated().sum()))

## Record decisions

Write source exclusions, privacy corrections, missing-date handling, and reviewed group merges in a dated report. Keep personal identifiers and unreviewed example text out of shared reports.